# [제출용 안내] WavLM-large seed 1 학습 기록

이 노트북은 최종 제출 모델의 **WavLM seed 1**(`checkpoints/m2_final_ensemble.pt` 안의 첫 번째 WavLM)을 만든 실제 실행 기록이다.
아래 셀들의 코드와 출력은 실행한 그대로이며, 이 안내 셀만 제출할 때 맨 위에 덧붙였다.

- **실행 이력**: 첫 Colab 계정에서 시작한 학습이 컴퓨팅 단위 소진으로 끊겼고, Drive 에는 같은 설정(seed 1)으로 학습한 step 1,000 체크포인트만 남았다.
  이 노트북은 새 계정(A100)에서 그 step 1,000 가중치로부터 이어서(옵티마이저는 새로 시작) 42,500 step 까지 학습했다(6번 셀, 151.2분).
  9번 셀의 전체 로그에는 첫 계정에서의 기록도 같은 로그 파일에 이어서 남아 있다.
- **모델 선택**: 7-2 셀에서 `best.pt`(step 15,000)와 마지막 step 모델을 떼어 둔 Training 발화 20,000개로 비교해 `best.pt`(92.73%)를 골랐다.
- **공식 Validation**: 7-3 셀, 92.3973% (평가 전용, 학습·선택에 쓰지 않음).
- 학습 코드는 제출 폴더의 `scripts/train_mission2_wavlm.py`, 모델 코드는 `src/dcc_data/mission2_wavlm.py` 와 같다
  (모델 코드는 이후 HuBERT 등 다른 사전학습 모델도 불러올 수 있게 일반화했고, WavLM 에 대해서는 출력이 완전히 같음을 확인했다).
- **9번 셀 로그의 `FAILED (exit 2)` 9줄에 대해**: 첫 계정에서 학습을 시작하기 전(10/12 08:41~09:07), GPU 속도를 재는 명령에 Drive 경로를
  따옴표로 감싸지 않아 경로의 띄어쓰기(`데이터 크리에이터 캠프`)에서 인자가 끊겨 명령이 거부된 기록이다(`unrecognized arguments`).
  경로를 고친 뒤 09:08부터 학습이 정상으로 시작됐고, 모델이나 결과에는 영향이 없다.


# Mission 2 — WavLM-large 미세조정 이어서 학습 (두 번째 Colab 세션)

첫 번째 세션(계정 rlarhkdgml12)이 Colab 컴퓨팅 단위 소진으로 끊겨서, **같은 학습을 새 계정에서 이어간다.**
- 같은 `RUN_ID` 폴더를 쓴다 → 학습 로그(`wavlm_log.txt`)가 같은 파일에 이어서 쌓이고, 제출용 노트북 마지막 셀에 전체 기록이 남는다.
- 이어갈 지점: Drive 에 저장된 `best.pt` / `snapshot.pt` 중 step 이 큰 파일 (첫 세션에서는 **step 24,000**, 그때의 12,000발화 점수 93.27%).
  가중치만 이어받고 optimizer 는 새로 시작한다(처음 200 step 은 학습률을 천천히 올림).
- 새 GPU 의 속도를 20 step 벤치마크로 재서, `TRAIN_HOURS` 안에 끝나도록 총 step 수를 정한다 (학습률은 그 끝에서 0이 되도록 줄어든다).
- 데이터/규정/분할은 첫 세션과 같다: 서울 Training 만, 제공 라벨 그대로, 발화 하나의 소리만, CNN 과 같은 통화 단위 80/20 분할.
  공식 Validation 은 학습과 모델 선택에 쓰지 않고, 마지막에 고른 모델 하나로 한 번만 잰다.

### 사용법
1. Colab **GPU 런타임**(A100 > L4 > T4 순으로 빠름)으로 설정하고 **런타임 > 모두 실행**.
2. 끊기면 `RUN_ID`를 그대로 두고 다시 "모두 실행" (같은 런타임이면 로컬 last.pt 에서, 새 런타임이면 Drive 의 snapshot.pt 에서 이어감).
3. 끝나면 마지막 셀 출력(전체 로그)까지 보이는 상태로 저장(Ctrl+S) 후 .ipynb 다운로드. **출력 지우기 금지.**

In [ ]:
# 1. 설정
RUN_ID = "wavlm_large_2026-10-12"   # 이어서 하므로 바꾸지 않는다 (로그와 체크포인트가 같은 폴더에 쌓임)
PRETRAINED = "microsoft/wavlm-large"
TRAIN_HOURS = 2.5        # 이번 세션에서 학습에 쓸 시간. GPU 가 느리면 step 수가 자동으로 줄어든다
MAX_TOTAL_STEPS = 66000  # 첫 세션 계획의 전체 step 수 (이보다 많이는 안 간다)
BATCH_SIZE = 32
FREEZE_LAYERS = 0
SEED = 1

DRIVE_ROOT = "/content/drive/MyDrive/데이터 크리에이터 캠프"     # 공유받은 폴더(바로가기)가 보이는 경로
CODE_DIR = f"{DRIVE_ROOT}/제출_wavlm/code"
ZIP_DIR = f"{DRIVE_ROOT}/대학부 데이터"
OUT_DIR = f"{DRIVE_ROOT}/m2_cache/m2_wavlm/{RUN_ID}"
DRIVE_MODEL = f"{OUT_DIR}/model"

DATA_DIR = "/content/dcc_data_full"
MANIFEST = "/content/dcc_manifest.csv"
LOCAL_MODEL = "/content/wavlm_local/model"   # 체크포인트는 로컬 디스크(빠름)에 저장

In [ ]:
# 2. Drive 마운트 + GPU 확인 + 로그 도구
import datetime, hashlib, json, os, re, shutil, subprocess, sys

from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(DRIVE_ROOT):
    raise SystemExit(f"{DRIVE_ROOT} 가 안 보임. '내 드라이브'에 있는 항목: {sorted(os.listdir('/content/drive/MyDrive'))[:20]} -- "
                     "공유받은 '데이터 크리에이터 캠프' 폴더를 '내 드라이브에 바로가기 추가' 했는지 확인")
assert os.path.isdir(OUT_DIR), f"{OUT_DIR} 없음 -- 첫 세션의 결과 폴더가 공유돼 있어야 이어갈 수 있다"
LOG_FILE = f"{OUT_DIR}/wavlm_log.txt"


def log(message: str) -> None:
    line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {message}"
    print(line, flush=True)
    with open(LOG_FILE, "a", encoding="utf-8") as stream:
        stream.write(line + "\n")


def run(cmd: str, *, cwd: str = "/content/prelim", capture: bool = False):
    """한 줄씩 셀과 로그 파일에 동시에 남긴다. capture=True 면 출력 줄 목록을 돌려준다."""
    log(f"$ {cmd}")
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}
    process = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=env)
    lines = []
    with open(LOG_FILE, "a", encoding="utf-8") as stream:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
    status = process.wait()
    if status != 0:
        log(f"FAILED (exit {status})")
        raise RuntimeError(f"명령 실패 (exit {status}): {cmd}")
    return lines if capture else None


def sha256(path: str) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


import torch
assert torch.cuda.is_available(), "런타임 > 런타임 유형 변경 > GPU(A100/L4/T4) 로 바꾸고 다시 시작할 것"
_gpu = torch.cuda.get_device_properties(0)
os.makedirs(LOCAL_MODEL, exist_ok=True)
os.makedirs(DRIVE_MODEL, exist_ok=True)
log("=" * 80)
log(f"세션 시작(이어서 학습): RUN_ID={RUN_ID} PRETRAINED={PRETRAINED} TRAIN_HOURS={TRAIN_HOURS} BATCH_SIZE={BATCH_SIZE}")
log(f"GPU={_gpu.name} ({_gpu.total_memory / 1e9:.0f}GB) torch={torch.__version__} CPU={os.cpu_count()}개 "
    f"디스크 여유={shutil.disk_usage('/content').free / 1e9:.0f}GB")

Mounted at /content/drive
[2026-10-12 14:31:41] ================================================================================
[2026-10-12 14:31:41] 세션 시작(이어서 학습): RUN_ID=wavlm_large_2026-10-12 PRETRAINED=microsoft/wavlm-large TRAIN_HOURS=2.5 BATCH_SIZE=32
[2026-10-12 14:31:41] GPU=NVIDIA A100-SXM4-40GB (42GB) torch=2.11.0+cu130 CPU=12개 디스크 여유=208GB


In [ ]:
# 3. 코드 준비 (Drive 의 코드 폴더) + 패키지 설치 + 코드 지문
assert os.path.isfile(f"{CODE_DIR}/scripts/train_mission2_wavlm.py"), f"코드 폴더가 없음: {CODE_DIR}"
_script = open(f"{CODE_DIR}/scripts/train_mission2_wavlm.py", encoding="utf-8").read()
assert "none yet" in _script, "Drive 의 train_mission2_wavlm.py 가 옛 버전임 -- 최신 파일로 덮어쓴 뒤 다시 실행"
if not os.path.isdir("/content/prelim"):
    shutil.copytree(CODE_DIR, "/content/prelim")
log(f"코드: {CODE_DIR}")
run("pip install -q -r requirements.txt")
run("pip list 2>/dev/null | grep -i -E '^(numpy|pandas|pyarrow|librosa|scikit-learn|lightgbm|tqdm|torch|transformers|soundfile) '")
_digest = hashlib.sha256()
for _dir, _subdirs, _files in sorted(os.walk("/content/prelim")):
    if "/." in _dir or "__pycache__" in _dir:
        continue
    for _name in sorted(f for f in _files if f.endswith(".py")):
        _path = os.path.join(_dir, _name)
        _digest.update(os.path.relpath(_path, "/content/prelim").encode())
        with open(_path, "rb") as _f:
            _digest.update(_f.read())
log(f"코드 지문(.py 전체 sha256)={_digest.hexdigest()}")

[2026-10-12 14:32:00] 코드: /content/drive/MyDrive/데이터 크리에이터 캠프/제출_wavlm/code
[2026-10-12 14:32:00] $ pip install -q -r requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 140.8 MB/s eta 0:00:00
[2026-10-12 14:32:14] $ pip list 2>/dev/null | grep -i -E '^(numpy|pandas|pyarrow|librosa|scikit-learn|lightgbm|tqdm|torch|transformers|soundfile) '
librosa                               0.11.0
lightgbm                              4.6.0
numpy                                 2.1.3
pandas                                2.2.3
pyarrow                               23.0.1
scikit-learn                          1.6.1
soundfile                             0.14.0
torch                                 2.11.0+cu130
tqdm                                  4.67.3
transformers                          5.19.0
[2026-10-12 14:32:16] 코드 지문(.py 전체 sha256)=f1eaac3593e87fbac4f212ba4315eed49a189331606ed758237e83b32f42039b


In [ ]:
# 4. 원본 zip -> 음성 풀기(서울 Training + Validation) -> 매니페스트  (새 런타임이라 다시 한다. 약 15분)
if not os.path.exists(MANIFEST):
    assert os.path.isdir(ZIP_DIR), f"원본 zip 폴더가 없음: {ZIP_DIR}"
    if not os.path.isdir(f"{DATA_DIR}/Training"):
        run(f"python scripts/extract_dcc_sample_from_zips.py --zip-dir '{ZIP_DIR}' --output-dir {DATA_DIR} --n-train-calls 0")
    run(f"python scripts/build_manifest.py --data-root {DATA_DIR} --output {MANIFEST} --overwrite")
else:
    log("데이터가 이미 준비돼 있음 (같은 런타임에서 다시 실행한 경우)")

[2026-10-12 14:32:16] $ python scripts/extract_dcc_sample_from_zips.py --zip-dir '/content/drive/MyDrive/데이터 크리에이터 캠프/대학부 데이터' --output-dir /content/dcc_data_full --n-train-calls 0
12개 zip에서 Seoul Training 전체 추출 중 (페어링 여부 무관, 이미 있는 파일은 건너뜀)...
Training 신규 추출 파일: 58400개
Validation 신규 추출 파일: 7280개 (전체, --n-val-calls 무시됨)
-> 실제로 몇 개 통화가 완전한 쌍으로 잡혔는지는 다음 단계(build_manifest.py)의 Training/Validation 행 수가 진짜 기준입니다 (여기 숫자는 파일 단위라 다를 수 있음).
이제 이 경로로 build_manifest.py를 실행하세요: /content/dcc_data_full
[2026-10-12 14:39:56] $ python scripts/build_manifest.py --data-root /content/dcc_data_full --output /content/dcc_manifest.csv --overwrite
Manifest: /content/dcc_manifest.csv
Rows: 32840
Training: 29200
Validation: 3640


## 5. 이어갈 지점 고르기 + 새 GPU 속도 재기
Drive 의 `snapshot.pt` / `best.pt` 중 step 이 큰 것에서 이어간다(정확도 기록은 지운 사본을 쓴다: 이번 세션의 첫 평가가 새 best 기준이 되도록).
20 step 벤치마크로 초당 속도를 재서 총 step 수를 `START_STEP + (TRAIN_HOURS 안에 갈 수 있는 step)`으로 정한다.

In [ ]:
# 5. 이어갈 파일 고르기 + 벤치마크
SETTINGS_FILE = f"{OUT_DIR}/train_settings_resume.json"


def step_of(path):
    return torch.load(path, map_location="cpu", weights_only=False)["step"]


if os.path.exists(f"{LOCAL_MODEL}/last.pt"):
    log("로컬 last.pt 가 있음 (같은 런타임에서 다시 실행) -> 거기서 이어감. 이전에 정한 설정을 그대로 사용")
    settings = json.load(open(SETTINGS_FILE, encoding="utf-8"))
    INIT_ARG = ""
else:
    candidates = {p: step_of(p) for p in (f"{DRIVE_MODEL}/snapshot.pt", f"{DRIVE_MODEL}/best.pt") if os.path.exists(p)}
    assert candidates, f"{DRIVE_MODEL} 에 best.pt / snapshot.pt 가 없음"
    log(f"Drive 의 이어갈 후보(step): {candidates}")
    source = max(candidates, key=candidates.get)
    start_step = candidates[source]
    init = torch.load(source, map_location="cpu", weights_only=False)
    init["internal_accuracy"] = None          # 이번 세션의 첫 평가가 새 best 의 기준이 된다
    torch.save(init, "/content/wavlm_init.pt")
    del init
    INIT_ARG = "--init-from /content/wavlm_init.pt"
    log(f"이어갈 파일: {source} (step {start_step})")

    base = (f"python scripts/train_mission2_wavlm.py --manifest {MANIFEST} --data-root {DATA_DIR} --pretrained {PRETRAINED} "
            f"--out-dir /content/wavlm_bench --seed {SEED} --freeze-layers {FREEZE_LAYERS} --amp --num-workers 2")
    settings = None
    for batch, ckpt_flag in ((BATCH_SIZE, ""), (BATCH_SIZE, " --grad-ckpt"), (BATCH_SIZE // 2, " --grad-ckpt")):
        try:
            lines = run(f"{base} --batch-size {batch}{ckpt_flag} --benchmark-steps 20", capture=True)
        except RuntimeError:
            log(f"batch {batch}{ckpt_flag}: 메모리 부족 등으로 실패 -- 다음 설정으로 재시도")
            continue
        bench = re.search(r"BENCH sec_per_step=([\d.]+) peak_gpu_gb=([\d.]+)", "".join(lines))
        sec_per_step = float(bench.group(1))
        # 평가와 저장이 1000 step 마다 약 15% 를 더 쓴다
        budget = int(TRAIN_HOURS * 3600 * 0.85 / sec_per_step)
        max_steps = min(MAX_TOTAL_STEPS, start_step + budget) // 500 * 500
        max_steps = max(max_steps, start_step + 3000)
        settings = {"batch_size": batch, "grad_ckpt": bool(ckpt_flag), "start_step": start_step, "max_steps": max_steps,
                    "sec_per_step": sec_per_step, "gpu": _gpu.name, "train_hours": TRAIN_HOURS}
        break
    assert settings, "세 가지 설정 모두 실패 -- FREEZE_LAYERS 를 8 로 올려서 다시 실행"
    json.dump(settings, open(SETTINGS_FILE, "w", encoding="utf-8"))
    log(f"설정 확정: {settings}")

[2026-10-12 15:30:00] Drive 의 이어갈 후보(step): {'/content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_wavlm/wavlm_large_2026-10-12/model/best.pt': 1000}
[2026-10-12 15:30:02] 이어갈 파일: /content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_wavlm/wavlm_large_2026-10-12/model/best.pt (step 1000)
[2026-10-12 15:30:02] $ python scripts/train_mission2_wavlm.py --manifest /content/dcc_manifest.csv --data-root /content/dcc_data_full --pretrained microsoft/wavlm-large --out-dir /content/wavlm_bench --seed 1 --freeze-layers 0 --amp --num-workers 2 --batch-size 32 --benchmark-steps 20
torch 2.11.0+cu130 transformers 5.19.0 device=cuda amp=True
train=711467 utterances / 23360 calls, held-out=177891 / 5840 calls (held-out calls sha256 8d59ba35c2f327e0)

Loading weights: 100%|██████████| 487/487 [00:00<00:00, 18878.24it/s]
[transformers] WavLMModel LOAD REPORT from: microsoft/wavlm-large
Key               | Status     |  | 
------------------+------------+--+-
masked_spec_embed | UNEXPECTED |  | 

Notes:
- UNEXPECT

## 6. 학습
- 체크포인트는 로컬 디스크에 저장하고, Drive 에는 `best.pt`(10분마다)와 `snapshot.pt`(최신 가중치, 30분마다)만 올린다.
- `step N/…` 줄은 50 step마다, `EVAL step N …` 줄은 1000 step마다(내부 검증 12,000발화, 0.5 고정 판정).

In [ ]:
# 6. 학습
SNAPSHOT_SCRIPT = """import os, sys, torch
LOCAL, DRIVE = sys.argv[1], sys.argv[2]
last_path = f"{LOCAL}/last.pt"
if not os.path.exists(last_path):
    sys.exit(0)
config_source = f"{LOCAL}/best.pt" if os.path.exists(f"{LOCAL}/best.pt") else f"{DRIVE}/best.pt"
last = torch.load(last_path, map_location="cpu", weights_only=False)
base = torch.load(config_source, map_location="cpu", weights_only=False)
snap = {**base, "state_dict": {k: (v.half() if v.is_floating_point() else v) for k, v in last["model"].items()},
        "step": last["step"], "internal_accuracy": None}
tmp = f"{DRIVE}/snapshot.pt.tmp"
torch.save(snap, tmp)
os.replace(tmp, f"{DRIVE}/snapshot.pt")
print("snapshot step", last["step"])
"""
with open("/content/wavlm_snapshot.py", "w", encoding="utf-8") as _f:
    _f.write(SNAPSHOT_SCRIPT)

for _marker in ("wavlm_best_sync", "wavlm_snapshot_sync"):
    try:
        subprocess.run(["pkill", "-f", _marker], check=False)
    except FileNotFoundError:
        pass
_best_sync = ("# wavlm_best_sync\nwhile true; do\n"
              f"  if [ -f {LOCAL_MODEL}/best.pt ]; then\n"
              f"    cp {LOCAL_MODEL}/best.pt '{DRIVE_MODEL}/best.pt.tmp' && mv -f '{DRIVE_MODEL}/best.pt.tmp' '{DRIVE_MODEL}/best.pt'\n"
              "  fi\n  sleep 600\ndone")
_snapshot_sync = ("# wavlm_snapshot_sync\nwhile true; do\n  sleep 1800\n"
                  f"  python /content/wavlm_snapshot.py {LOCAL_MODEL} '{DRIVE_MODEL}'\ndone")
for _script in (_best_sync, _snapshot_sync):
    subprocess.Popen(["bash", "-c", _script], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

TRAIN_ARGS = (
    f"python scripts/train_mission2_wavlm.py --manifest {MANIFEST} --data-root {DATA_DIR} --pretrained {PRETRAINED} "
    f"--out-dir {LOCAL_MODEL} --seed {SEED} --freeze-layers {FREEZE_LAYERS} --amp --num-workers 2 "
    f"--batch-size {settings['batch_size']}{' --grad-ckpt' if settings['grad_ckpt'] else ''} "
    f"--max-steps {settings['max_steps']} --eval-every 1000 --save-every 1000 --resume {INIT_ARG}"
)
run(TRAIN_ARGS)

# 학습이 끝나면 최종 best.pt 를 Drive 에 확실히 올린다
shutil.copyfile(f"{LOCAL_MODEL}/best.pt", f"{DRIVE_MODEL}/best.pt.tmp")
os.replace(f"{DRIVE_MODEL}/best.pt.tmp", f"{DRIVE_MODEL}/best.pt")
log(f"best.pt sha256={sha256(f'{DRIVE_MODEL}/best.pt')}")

[2026-10-12 15:30:30] $ python scripts/train_mission2_wavlm.py --manifest /content/dcc_manifest.csv --data-root /content/dcc_data_full --pretrained microsoft/wavlm-large --out-dir /content/wavlm_local/model --seed 1 --freeze-layers 0 --amp --num-workers 2 --batch-size 32 --max-steps 42500 --eval-every 1000 --save-every 1000 --resume --init-from /content/wavlm_init.pt
torch 2.11.0+cu130 transformers 5.19.0 device=cuda amp=True
train=711467 utterances / 23360 calls, held-out=177891 / 5840 calls (held-out calls sha256 8d59ba35c2f327e0)

Loading weights: 100%|██████████| 487/487 [00:00<00:00, 17684.92it/s]
[transformers] WavLMModel LOAD REPORT from: microsoft/wavlm-large
Key               | Status     |  | 
------------------+------------+--+-
masked_spec_embed | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
pretrained=microsoft/wavlm-large (commit None) parameters: total 315,977,179
trainable param

## 7. 마무리: 마지막 모델 저장 -> 두 모델 비교 -> 공식 Validation 한 번
- `best.pt`는 "1000 step마다 잰 12,000발화 점수 중 최고"라 운 좋은 봉우리일 수 있고, 학습 후반(학습률 0 근처)의 마지막 모델 `final.pt`가 더 나을 수 있다.
- 둘을 **학습에 안 쓴 새 표본 20,000발화**로 다시 비교해 고르고, 공식 Validation 은 고른 모델 하나로 **한 번만** 잰다(평가 전용).
- Colab 에서는 속도를 위해 float16(`--amp`)으로 확률을 낸다. 최종 점수는 나중에 float32 로 표본 대조한다.

In [ ]:
# 7-1. 마지막 step 모델을 best.pt 와 같은 형식(final.pt)으로 저장
_best = torch.load(f"{LOCAL_MODEL}/best.pt", map_location="cpu", weights_only=False)
_last = torch.load(f"{LOCAL_MODEL}/last.pt", map_location="cpu", weights_only=False)
_final = {**_best, "state_dict": {k: (v.half() if v.is_floating_point() else v) for k, v in _last["model"].items()},
          "step": _last["step"], "internal_accuracy": None}
torch.save(_final, f"{LOCAL_MODEL}/final.pt")
log(f"final.pt 저장: step {_final['step']} (best.pt 는 step {_best['step']}, 12,000발화 점수 {_best['internal_accuracy']:.4f})")
del _best, _last, _final

[2026-10-12 18:02:13] final.pt 저장: step 42500 (best.pt 는 step 15000, 12,000발화 점수 0.9287)


In [ ]:
# 7-2. 내부 검증 새 표본 20,000발화로 best.pt 와 final.pt 비교 (공식 Validation 은 아직 안 씀)
_scores = {}
for _name in ("best", "final"):
    _lines = run(
        f"python scripts/predict_mission2_wavlm.py --manifest {MANIFEST} --data-root {DATA_DIR} "
        f"--ckpt {LOCAL_MODEL}/{_name}.pt --split internal --utts 20000 --amp "
        f"--output '{OUT_DIR}/wavlm_internal_probs_{_name}.csv' --num-workers 2",
        capture=True,
    )
    _scores[_name] = float(re.search(r"internal accuracy at 0\.5: ([\d.]+)%", "".join(_lines)).group(1))
CHOSEN = max(_scores, key=_scores.get)  # 동점이면 best
log(f"내부 20,000발화 정확도: {_scores} -> 선택: {CHOSEN}.pt")
for _name in ("best", "final"):
    shutil.copyfile(f"{LOCAL_MODEL}/{_name}.pt", f"{DRIVE_MODEL}/{_name}.pt.tmp")
    os.replace(f"{DRIVE_MODEL}/{_name}.pt.tmp", f"{DRIVE_MODEL}/{_name}.pt")
    log(f"{_name}.pt sha256={sha256(f'{DRIVE_MODEL}/{_name}.pt')}")
with open(f"{OUT_DIR}/chosen_model.json", "w", encoding="utf-8") as _f:
    json.dump({"chosen": CHOSEN, "internal_20000_accuracy": _scores}, _f)

[2026-10-12 18:02:13] $ python scripts/predict_mission2_wavlm.py --manifest /content/dcc_manifest.csv --data-root /content/dcc_data_full --ckpt /content/wavlm_local/model/best.pt --split internal --utts 20000 --amp --output '/content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_wavlm/wavlm_large_2026-10-12/wavlm_internal_probs_best.csv' --num-workers 2
internal: 20000 utterances / 5517 calls, device=cuda
  6400/20000
  12800/20000
  19200/20000
internal accuracy at 0.5: 92.7300% (18546/20000), recall_0 0.8834 recall_1 0.9677; 0.7 min, 2.0 ms per utterance, batch 64
wrote /content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_wavlm/wavlm_large_2026-10-12/wavlm_internal_probs_best.csv
[2026-10-12 18:03:28] $ python scripts/predict_mission2_wavlm.py --manifest /content/dcc_manifest.csv --data-root /content/dcc_data_full --ckpt /content/wavlm_local/model/final.pt --split internal --utts 20000 --amp --output '/content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_wavlm/wavlm_large_2026-10-12/wavlm_internal_pro

In [ ]:
# 7-3. 공식 Validation 전체 확률 -- 고른 모델 하나로 한 번만 (평가 전용)
run(f"python scripts/predict_mission2_wavlm.py --manifest {MANIFEST} --data-root {DATA_DIR} "
    f"--ckpt {LOCAL_MODEL}/{CHOSEN}.pt --split official --amp "
    f"--output '{OUT_DIR}/wavlm_official_probs.csv' --num-workers 2")

[2026-10-12 18:04:40] $ python scripts/predict_mission2_wavlm.py --manifest /content/dcc_manifest.csv --data-root /content/dcc_data_full --ckpt /content/wavlm_local/model/best.pt --split official --amp --output '/content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_wavlm/wavlm_large_2026-10-12/wavlm_official_probs.csv' --num-workers 2
official: 111947 utterances / 3640 calls, device=cuda
  6400/111947
  12800/111947
  19200/111947
  25600/111947
  32000/111947
  38400/111947
  44800/111947
  51200/111947
  57600/111947
  64000/111947
  70400/111947
  76800/111947
  83200/111947
  89600/111947
  96000/111947
  102400/111947
  108800/111947
official accuracy at 0.5: 92.3973% (103436/111947), recall_0 0.8803 recall_1 0.9645; 3.6 min, 1.9 ms per utterance, batch 64
wrote /content/drive/MyDrive/데이터 크리에이터 캠프/m2_cache/m2_wavlm/wavlm_large_2026-10-12/wavlm_official_probs.csv


In [ ]:
# 8. 요약
_ck = torch.load(f"{LOCAL_MODEL}/{CHOSEN}.pt", map_location="cpu", weights_only=False)
_params = sum(v.numel() for v in _ck["state_dict"].values())
log(f"선택한 모델: {CHOSEN}.pt (step {_ck['step']}), 사전학습 {_ck['pretrained']} (commit {_ck['pretrained_commit']})")
log(f"파라미터 수 {_params:,}개 (backbone + 층 가중치 + 분류 head), 크기 {os.path.getsize(f'{LOCAL_MODEL}/{CHOSEN}.pt') / 1e6:.0f}MB (float16)")
log(f"이번 세션 학습 GPU: {_gpu.name}, 설정: {settings}")

[2026-10-12 18:08:34] 선택한 모델: best.pt (step 15000), 사전학습 microsoft/wavlm-large (commit None)
[2026-10-12 18:08:34] 파라미터 수 315,977,179개 (backbone + 층 가중치 + 분류 head), 크기 632MB (float16)
[2026-10-12 18:08:34] 이번 세션 학습 GPU: NVIDIA A100-SXM4-40GB, 설정: {'batch_size': 32, 'grad_ckpt': False, 'start_step': 1000, 'max_steps': 42500, 'sec_per_step': 0.184, 'gpu': 'NVIDIA A100-SXM4-40GB', 'train_hours': 2.5}


In [ ]:
# 9. 로그 파일 전체 출력 -- 두 세션의 기록이 이어진 전체 로그 (출력 지우기 금지)
with open(LOG_FILE, encoding="utf-8") as stream:
    print(stream.read())

[2026-10-12 08:33:35] ================================================================================
[2026-10-12 08:33:35] 새 실행 시작: RUN_ID=wavlm_large_2026-10-12 PRETRAINED=microsoft/wavlm-large TRAIN_HOURS=8.0 BATCH_SIZE=32 FREEZE_LAYERS=0 SEED=1
[2026-10-12 08:33:35] GPU=NVIDIA A100-SXM4-40GB (42GB) torch=2.11.0+cu130 CPU=12개 디스크 여유=208GB
[2026-10-12 08:33:35] Drive 여유=197GB (체크포인트가 약 5GB 필요)
[2026-10-12 08:34:16] 코드: /content/drive/MyDrive/데이터 크리에이터 캠프/제출_wavlm/code
[2026-10-12 08:34:16] $ pip install -q -r requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 124.6 MB/s eta 0:00:00
[2026-10-12 08:34:30] $ pip list 2>/dev/null | grep -i -E '^(numpy|pandas|pyarrow|librosa|scikit-learn|lightgbm|tqdm|torch|transformers|soundfile) '
librosa                               0.11.0
lightgbm                              4.6.0
numpy                                 2.1.3
pandas                                2.2.3
pyarrow                               23.0.1
scikit-learn  